In [9]:
from __future__ import annotations

import urllib.parse
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine
from IPython.display import display

# =========================
# 0) 고정 설정
# =========================
KST = ZoneInfo("Asia/Seoul")

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

PG_WORK_MEM = "64MB"

SRC_SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
SRC_TABLE  = "fct_vision_testlog_txt_processing_history"
SRC_FQN = f"{SRC_SCHEMA}.{SRC_TABLE}"

STATIONS = ["FCT1", "FCT2", "FCT3", "FCT4", "Vision1", "Vision2"]

# 집계 대상
PROD_DAY = "20260130"  # 예: 20260130

# =========================
# 1) 엔진 (비번 포함, 특수문자 안전)
# =========================
def make_engine() -> Engine:
    user = urllib.parse.quote_plus(DB_CONFIG["user"])
    pw   = urllib.parse.quote_plus(DB_CONFIG["password"])
    host = DB_CONFIG["host"]
    port = DB_CONFIG["port"]
    db   = DB_CONFIG["dbname"]

    url = f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}"
    return create_engine(
        url,
        pool_pre_ping=True,
        pool_size=1,
        max_overflow=0,
        pool_recycle=1800,
    )

def set_session_params(conn) -> None:
    conn.execute(text(f"SET work_mem = '{PG_WORK_MEM}'"))
    conn.execute(text("SET client_encoding = 'UTF8'"))

# =========================
# 2) 윈도우 (end_day + end_time 기준)
# =========================
def shift_window(prod_day: str, shift_type: str) -> tuple[datetime, datetime]:
    d0 = datetime.strptime(prod_day, "%Y%m%d").replace(tzinfo=KST)

    if shift_type == "day":
        start = d0.replace(hour=8, minute=30, second=0)
        end   = d0.replace(hour=20, minute=29, second=59)
        return start, end

    if shift_type == "night":
        start = d0.replace(hour=20, minute=30, second=0)
        d1 = d0 + timedelta(days=1)
        end = d1.replace(hour=8, minute=29, second=59)
        return start, end

    raise ValueError("shift_type must be 'day' or 'night'")

def build_window_agg_query(prod_day: str, shift_type: str) -> tuple[str, dict]:
    start_dt, end_dt = shift_window(prod_day, shift_type)

    sql = f"""
    WITH base AS (
      SELECT station, result
      FROM {SRC_FQN}
      WHERE
        station = ANY(:stations)
        AND result IN ('PASS','FAIL')
        AND to_timestamp(end_day || ' ' || end_time, 'YYYYMMDD HH24:MI:SS')
            BETWEEN :start_dt AND :end_dt
    )
    SELECT
      station,
      SUM(CASE WHEN result='PASS' THEN 1 ELSE 0 END)::int AS pass_cnt,
      SUM(CASE WHEN result='FAIL' THEN 1 ELSE 0 END)::int AS fail_cnt,
      COUNT(*)::int AS total_cnt
    FROM base
    GROUP BY station
    ORDER BY station
    """

    params = {
        "stations": STATIONS,
        # DB 비교가 timestamp(no tz)라서 파라미터도 naive로 전달
        "start_dt": start_dt.replace(tzinfo=None),
        "end_dt": end_dt.replace(tzinfo=None),
    }
    return sql, params

# =========================
# 3) 결과 DF (셀 문자열 포맷 + updated_at)
# =========================
def fmt_cell(pass_cnt: int, total_cnt: int) -> str:
    pct = (pass_cnt / total_cnt * 100.0) if total_cnt > 0 else 0.0
    return f"PASS: {pass_cnt}, total: {total_cnt}, PASS_pct:{pct:.2f}"

def build_result_df(prod_day: str, shift_type: str) -> pd.DataFrame:
    engine = make_engine()
    sql, params = build_window_agg_query(prod_day, shift_type)

    with engine.begin() as conn:
        set_session_params(conn)
        rows = conn.execute(text(sql), params).mappings().all()

    agg = {s: {"pass": 0, "total": 0} for s in STATIONS}
    for r in rows:
        s = r.get("station")
        if s in agg:
            agg[s]["pass"]  = int(r.get("pass_cnt") or 0)
            agg[s]["total"] = int(r.get("total_cnt") or 0)

    # Vision1 + Vision2 합산
    v_pass  = agg["Vision1"]["pass"]  + agg["Vision2"]["pass"]
    v_total = agg["Vision1"]["total"] + agg["Vision2"]["total"]

    updated_at = datetime.now(KST).strftime("%Y-%m-%d %H:%M:%S%z")

    out = {
        "prod_day": prod_day,
        "shift_type": shift_type,
        "FCT1": fmt_cell(agg["FCT1"]["pass"], agg["FCT1"]["total"]),
        "FCT2": fmt_cell(agg["FCT2"]["pass"], agg["FCT2"]["total"]),
        "FCT3": fmt_cell(agg["FCT3"]["pass"], agg["FCT3"]["total"]),
        "FCT4": fmt_cell(agg["FCT4"]["pass"], agg["FCT4"]["total"]),
        "Vision1": fmt_cell(agg["Vision1"]["pass"], agg["Vision1"]["total"]),
        "Vision2": fmt_cell(agg["Vision2"]["pass"], agg["Vision2"]["total"]),
        "FCT>Vision Total": fmt_cell(v_pass, v_total),   # ✅ 컬럼명 변경
        "updated_at": updated_at,
    }
    return pd.DataFrame([out])

# =========================
# 4) 실행: day / night 따로 출력
# =========================
engine = make_engine()
with engine.begin() as conn:
    print("DB OK:", conn.execute(text("SELECT 1")).scalar())

day_df = build_result_df(PROD_DAY, "day")
night_df = build_result_df(PROD_DAY, "night")

print(f"[RESULT] prod_day={PROD_DAY} shift=day")
display(day_df)

print(f"[RESULT] prod_day={PROD_DAY} shift=night")
display(night_df)

DB OK: 1
[RESULT] prod_day=20260130 shift=day


,prod_day,shift_type,FCT1,FCT2,FCT3,FCT4,Vision1,Vision2,FCT>Vision Total,updated_at
0,20260130,day,"PASS: 832, total: 899, PASS_pct:92.55","PASS: 896, total: 906, PASS_pct:98.90","PASS: 971, total: 978, PASS_pct:99.28","PASS: 987, total: 992, PASS_pct:99.50","PASS: 1736, total: 1741, PASS_pct:99.71","PASS: 1954, total: 1960, PASS_pct:99.69","PASS: 3690, total: 3701, PASS_pct:99.70",2026-01-31 12:03:58+0900


[RESULT] prod_day=20260130 shift=night


,prod_day,shift_type,FCT1,FCT2,FCT3,FCT4,Vision1,Vision2,FCT>Vision Total,updated_at
0,20260130,night,"PASS: 861, total: 959, PASS_pct:89.78","PASS: 982, total: 993, PASS_pct:98.89","PASS: 884, total: 932, PASS_pct:94.85","PASS: 909, total: 944, PASS_pct:96.29","PASS: 1833, total: 1833, PASS_pct:100.00","PASS: 1793, total: 1802, PASS_pct:99.50","PASS: 3626, total: 3635, PASS_pct:99.75",2026-01-31 12:04:05+0900


In [11]:
from sqlalchemy import text
import pandas as pd

SAVE_SCHEMA = "i_daily_report"
DAY_TABLE   = "b_station_day_daily_percentage"
NIGHT_TABLE = "b_station_night_daily_percentage"
KEY_COLS = ["prod_day", "shift_type"]

def ensure_schema(engine, schema: str) -> None:
    with engine.begin() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema};"))

def ensure_table(engine, schema: str, table: str, df: pd.DataFrame, key_cols: list[str]) -> None:
    cols = list(df.columns)

    with engine.begin() as conn:
        exists = conn.execute(text("""
            SELECT EXISTS (
              SELECT 1
              FROM information_schema.tables
              WHERE table_schema = :schema AND table_name = :table
            );
        """), {"schema": schema, "table": table}).scalar()

        if not exists:
            ddl_cols = []
            for c in cols:
                if c == "updated_at":
                    ddl_cols.append(f'"{c}" timestamptz')
                else:
                    ddl_cols.append(f'"{c}" text')

            ddl_cols_sql = ",\n  ".join(ddl_cols)
            uq_sql = ", ".join([f'"{c}"' for c in key_cols])

            conn.execute(text(f"""
                CREATE TABLE {schema}.{table} (
                  {ddl_cols_sql},
                  CONSTRAINT uq_{table}_key UNIQUE ({uq_sql})
                );
            """))
        else:
            existing_cols = conn.execute(text("""
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = :schema AND table_name = :table
            """), {"schema": schema, "table": table}).fetchall()
            existing_cols = {r[0] for r in existing_cols}

            for c in cols:
                if c in existing_cols:
                    continue
                col_type = "timestamptz" if c == "updated_at" else "text"
                conn.execute(text(f'ALTER TABLE {schema}.{table} ADD COLUMN "{c}" {col_type};'))

def upsert_df(engine, schema: str, table: str, df: pd.DataFrame, key_cols: list[str]) -> None:
    """
    ✅ 컬럼명이 'FCT>Vision Total'처럼 특수문자를 포함해도 안전하게 UPSERT
    - 바인드 파라미터는 v0, v1...로 따로 만든다.
    - updated_at은 timestamptz 컬럼이므로 파이썬 datetime(tz-aware)로 넣는 걸 권장.
    """
    assert len(df) == 1, "day_df/night_df (1행 DF)만 저장하도록 설계됨"

    row0 = df.iloc[0].to_dict()
    cols = list(df.columns)

    # 1) 안전한 bind key 생성
    bind_map = {c: f"v{i}" for i, c in enumerate(cols)}
    params = {bind_map[c]: row0.get(c) for c in cols}

    # updated_at이 문자열이면 tz-aware datetime으로 바꿔주는게 가장 안전
    # (현재 너 DF는 "2026-01-31 12:03:58+0900" 형태라서 파싱 가능)
    if "updated_at" in cols and isinstance(params[bind_map["updated_at"]], str):
        params[bind_map["updated_at"]] = pd.to_datetime(params[bind_map["updated_at"]]).to_pydatetime()

    insert_cols_sql = ", ".join([f'"{c}"' for c in cols])
    values_sql = ", ".join([f":{bind_map[c]}" for c in cols])

    conflict_cols_sql = ", ".join([f'"{c}"' for c in key_cols])

    set_parts = []
    for c in cols:
        if c in key_cols:
            continue
        if c == "updated_at":
            set_parts.append(f'"{c}" = EXCLUDED."{c}"')
        else:
            set_parts.append(f'"{c}" = COALESCE(EXCLUDED."{c}", {schema}.{table}."{c}")')
    set_sql = ", ".join(set_parts)

    sql = f"""
        INSERT INTO {schema}.{table} ({insert_cols_sql})
        VALUES ({values_sql})
        ON CONFLICT ({conflict_cols_sql})
        DO UPDATE SET {set_sql};
    """

    with engine.begin() as conn:
        conn.execute(text(sql), params)

# =========================
# 실행: day_df / night_df 저장
# =========================
engine = make_engine()

ensure_schema(engine, SAVE_SCHEMA)

# day 저장
ensure_table(engine, SAVE_SCHEMA, DAY_TABLE, day_df, KEY_COLS)
upsert_df(engine, SAVE_SCHEMA, DAY_TABLE, day_df, KEY_COLS)
print(f"[SAVED] {SAVE_SCHEMA}.{DAY_TABLE}  prod_day={day_df.at[0,'prod_day']} shift={day_df.at[0,'shift_type']}")

# night 저장
ensure_table(engine, SAVE_SCHEMA, NIGHT_TABLE, night_df, KEY_COLS)
upsert_df(engine, SAVE_SCHEMA, NIGHT_TABLE, night_df, KEY_COLS)
print(f"[SAVED] {SAVE_SCHEMA}.{NIGHT_TABLE} prod_day={night_df.at[0,'prod_day']} shift={night_df.at[0,'shift_type']}")

[SAVED] i_daily_report.b_station_day_daily_percentage  prod_day=20260130 shift=day
[SAVED] i_daily_report.b_station_night_daily_percentage prod_day=20260130 shift=night
